In [23]:
import cv2
import face_recognition
import serial
import time
import pickle

# Load teacher encoding
with open("teacher_face.pkl", "rb") as f:
    teacher_encoding = pickle.load(f)

# Connect to Arduino
arduino = serial.Serial("COM11", 9600)
time.sleep(2)

cap = cv2.VideoCapture(1)
servo_angle = 90  # Start at center
searching_direction = 1  # 1 = right, -1 = left

def clamp(val, min_val=0, max_val=180):
    return max(min_val, min(val, max_val))

def map_value(x, in_min, in_max, out_min, out_max):
    return int((x - in_min) * (out_max - out_min) / (in_max - in_min) + out_min)

last_seen = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)
    height, width = frame.shape[:2]
    center_x = width // 2
    tolerance = 40

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    boxes = face_recognition.face_locations(rgb)
    encodings = face_recognition.face_encodings(rgb, boxes)

    found = False

    for (box, encoding) in zip(boxes, encodings):
        match = face_recognition.compare_faces([teacher_encoding], encoding, tolerance=0.5)
        if match[0]:
            found = True
            last_seen = time.time()

            (top, right, bottom, left) = box
            face_center = (left + right) // 2

            # Visual feedback
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
            cv2.putText(frame, "Teacher", (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            # Centered check
            if face_center < center_x - tolerance:
                servo_angle -= 2
            elif face_center > center_x + tolerance:
                servo_angle += 2
            else:
                # Face is centered → do not move
                pass

            servo_angle = clamp(servo_angle)
            arduino.write(f"{servo_angle}\n".encode())
            break

    # If teacher not seen recently → keep scanning
    if not found and (time.time() - last_seen > 1.0):
        servo_angle += searching_direction
        if servo_angle >= 180 or servo_angle <= 0:
            searching_direction *= -1  # reverse direction
            servo_angle = clamp(servo_angle)
        arduino.write(f"{servo_angle}\n".encode())
        time.sleep(0.02)

    cv2.line(frame, (center_x, 0), (center_x, height), (255, 0, 0), 2)
    cv2.imshow("Smart Face Tracker", frame)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
